In [ ]:
# Cell 1 — Mount Drive
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# Cell 2 — Install dependencies
%pip install anthropic deepeval jsonlines tqdm pandas tiktoken sentence-transformers -q


In [ ]:
# Cell 3 — Load config + API key
import pathlib
import sys

p = pathlib.Path.cwd().resolve()
while p != p.parent and not (p / "synthetic_data").exists():
    p = p.parent
sys.path.append(str(p))

from synthetic_data.colab.config import *

from google.colab import userdata
import anthropic

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
print("Setup complete ✓")


In [ ]:
# Cell 4 — Checkpoint helper
import os

import jsonlines


def save_checkpoint(data, filename):
    path = f"{SYNTHETIC_DIR}/{filename}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with jsonlines.open(path, "w") as w:
        w.write_all(data)
    print(f"Saved {len(data)} items -> {path} ✓")


def load_checkpoint(filename):
    path = f"{SYNTHETIC_DIR}/{filename}"
    if not os.path.exists(path):
        return []
    with jsonlines.open(path) as r:
        return list(r)


In [ ]:
from synthetic_data.pipeline.embedding_triplets import generate_embedding_triplets

# ── RUN ────────────────────────────────────────────────────────
triplets = generate_embedding_triplets(
    synthetic_dir=SYNTHETIC_DIR,
    client=client,
    model=CLAUDE_MODEL,
    retry_limit=RETRY_LIMIT,
    target_total=EMBEDDING_TARGET,
)
print(f"Total triplets generated (raw): {len(triplets)}")
